# Resort Churn — DecisionTreeClassifier + Optuna

Minimal-preprocessing `DecisionTreeClassifier` with Optuna hyperparameter tuning.

- **Train:** `resort_train.csv` (from the 869_course repo)
- **Test:** `resort_test.csv`
- **Metric:** macro F1 (mean over 5-fold stratified CV)
- **Tuning:** Optuna maximizes the mean CV macro F1; the best hyperparameters are reported.
- **Output:** `submission.csv` with `GuestID` and 0/1 `Churned` predictions

Rather than discarding `BookingDate` and `Room`, both are **feature-engineered** into
model-friendly columns:

- `BookingDate` → calendar parts (year, month, day, day-of-week, quarter, day-of-year,
  week-of-year, weekend flag).
- `Room` (`"<Wing>/<RoomNumber>/<Suffix>"`) → wing letter, numeric room number, suffix/class
  letter, and a leading-digit floor proxy.

The rest of the preprocessing stays minimal: drop the ID column, median-impute numerics,
and one-hot encode the low-cardinality categoricals.

Performance is reported as the **mean** CV macro F1 and an **out-of-fold** classification
report (every training row scored once, on the fold where it was held out) — this reflects
average generalization rather than a single lucky fold.

In [1]:
import pandas as pd
import numpy as np
import optuna

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import classification_report

RANDOM_STATE = 42
optuna.logging.set_verbosity(optuna.logging.WARNING)  # keep the trial log quiet

In [2]:
# Load data straight from the course repo (raw URLs)
TRAIN_URL = "https://raw.githubusercontent.com/stepthom/869_course/main/data/resort_train.csv"
TEST_URL = "https://raw.githubusercontent.com/stepthom/869_course/main/data/resort_test.csv"

train = pd.read_csv(TRAIN_URL)
test = pd.read_csv(TEST_URL)

print("train:", train.shape, "| test:", test.shape)
train.head()

train: (6954, 21) | test: (1739, 20)


,GuestID,BookingDate,PromoCode,Region,AllInclusive,Room,PackageType,Age,VIP,RoomService,...,Retail,Spa,Entertainment,LoyaltyPoints,SurveyScore,DaysSinceEmail,BookingChannel,AgeGroup,ReferralSource,Churned
0,619623,2024-02-10,NaN,Americas,0.0,G/630/S,Relaxation,0.0,0.0,0.0,...,0.0,0.0,0.0,6915,5,136,Website,NaN,Facebook,1
1,776250,2024-01-03,NaN,Americas,1.0,G/201/S,Relaxation,17.0,0.0,0.0,...,0.0,0.0,0.0,8571,5,362,Corporate,Minor,Billboard,1
2,932709,2023-01-17,NaN,Americas,NaN,G/1483/S,Wellness,35.0,0.0,0.0,...,0.0,0.0,0.0,1142,4,154,TravelAgent,Middle,Facebook,0
3,771839,2023-12-09,PromoA,Europe,1.0,D/164/S,Adventure,26.0,0.0,0.0,...,0.0,NaN,0.0,9642,2,128,Website,Young,Magazine,1
4,755501,2024-02-15,PromoA,Americas,0.0,G/818/P,Relaxation,13.0,0.0,0.0,...,60.0,1.0,5147.0,5528,4,35,Mobile,Minor,Google,0


In [3]:
TARGET = "Churned"
ID_COL = "GuestID"

# Instead of dropping BookingDate and Room, split them into useful columns.
#
# BookingDate (e.g. "2024-02-10") -> calendar parts the model can split on.
# Room (e.g. "G/630/S") is "<Wing>/<RoomNumber>/<Suffix>":
#   - Wing:   building/wing letter (categorical, low cardinality)
#   - RoomNumber: integer; its leading digit is a rough floor proxy
#   - Suffix: room class letter, e.g. S / P (categorical, low cardinality)
def engineer_features(df):
    df = df.copy()

    # --- BookingDate -> calendar features ---
    bd = pd.to_datetime(df["BookingDate"], errors="coerce")
    df["Booking_Year"] = bd.dt.year
    df["Booking_Month"] = bd.dt.month
    df["Booking_Day"] = bd.dt.day
    df["Booking_DayOfWeek"] = bd.dt.dayofweek
    df["Booking_Quarter"] = bd.dt.quarter
    df["Booking_DayOfYear"] = bd.dt.dayofyear
    df["Booking_WeekOfYear"] = bd.dt.isocalendar().week.astype("float")
    df["Booking_IsWeekend"] = (bd.dt.dayofweek >= 5).astype("float")

    # --- Room -> wing / number / suffix ---
    room_parts = df["Room"].str.split("/", expand=True)
    df["Room_Wing"] = room_parts[0]
    df["Room_Number"] = pd.to_numeric(room_parts[1], errors="coerce")
    df["Room_Suffix"] = room_parts[2]
    # Leading digit of the room number as a coarse floor/zone proxy.
    df["Room_Floor"] = pd.to_numeric(
        room_parts[1].str.replace(r"\D", "", regex=True).str[0], errors="coerce"
    )

    return df.drop(columns=["BookingDate", "Room"])


train_fe = engineer_features(train)
test_fe = engineer_features(test)

# Drop only the identifier now; the engineered columns stay in.
X = train_fe.drop(columns=[ID_COL, TARGET])
y = train_fe[TARGET]
X_test = test_fe.drop(columns=[ID_COL])

numeric_cols = X.select_dtypes(include="number").columns.tolist()
categorical_cols = X.select_dtypes(exclude="number").columns.tolist()

print("numeric:", numeric_cols)
print("categorical:", categorical_cols)

numeric: ['AllInclusive', 'Age', 'VIP', 'RoomService', 'Dining', 'Retail', 'Spa', 'Entertainment', 'LoyaltyPoints', 'SurveyScore', 'DaysSinceEmail', 'Booking_Year', 'Booking_Month', 'Booking_Day', 'Booking_DayOfWeek', 'Booking_Quarter', 'Booking_DayOfYear', 'Booking_WeekOfYear', 'Booking_IsWeekend', 'Room_Number', 'Room_Floor']
categorical: ['PromoCode', 'Region', 'PackageType', 'BookingChannel', 'AgeGroup', 'ReferralSource', 'Room_Wing', 'Room_Suffix']


In [4]:
# Minimal preprocessing: impute then one-hot encode categoricals.
# DecisionTree can't take NaNs or strings, so this is the smallest transform that works.
preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_cols),
        (
            "cat",
            Pipeline([
                ("impute", SimpleImputer(strategy="most_frequent")),
                ("ohe", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_cols,
        ),
    ]
)


def make_model(**params):
    """Full pipeline: shared preprocessing + a DecisionTree with the given hyperparameters."""
    return Pipeline([
        ("prep", preprocess),
        ("clf", DecisionTreeClassifier(random_state=RANDOM_STATE, **params)),
    ])


# Shared CV splitter used for both tuning and reporting.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

In [5]:
# Optuna hyperparameter tuning.
# Objective: maximize the MEAN macro F1 across the 5 CV folds.
N_TRIALS = 40


def objective(trial):
    # DecisionTreeClassifier hyperparameters to search.
    params = {
        "max_depth": trial.suggest_int("max_depth", 5, 40),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "criterion": trial.suggest_categorical("criterion", ["gini", "entropy"]),
    }
    candidate = make_model(**params)
    scores = cross_val_score(candidate, X, y, cv=cv, scoring="f1_macro")
    return scores.mean()


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

best_params = study.best_params
print(f"Trials run: {len(study.trials)}")
print(f"Best mean CV macro F1: {study.best_value:.4f}")
print("Best hyperparameters:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

Trials run: 40
Best mean CV macro F1: 0.7596
Best hyperparameters:
  max_depth: 38
  min_samples_split: 20
  min_samples_leaf: 10
  max_features: None
  criterion: entropy


In [6]:
# Build the tuned model and evaluate it with the same 5-fold stratified CV.
model = make_model(**best_params)

scores = cross_val_score(model, X, y, cv=cv, scoring="f1_macro")
print("Macro F1 per fold:", np.round(scores, 4))
print(f"Mean macro F1: {scores.mean():.4f} (+/- {scores.std():.4f})")

# Out-of-fold predictions: each row is predicted by the fold in which it was held out,
# so a single classification report reflects mean CV performance (not one lucky fold).
oof_pred = cross_val_predict(model, X, y, cv=cv)

Macro F1 per fold: [0.7656 0.7634 0.7597 0.7584 0.7511]
Mean macro F1: 0.7596 (+/- 0.0050)


In [7]:
# Out-of-fold classification report for the tuned model (mean CV performance).
print("Out-of-fold classification report (tuned DecisionTree):\n")
print(classification_report(y, oof_pred, digits=4))

Out-of-fold classification report (tuned DecisionTree):

              precision    recall  f1-score   support

           0     0.7473    0.7795    0.7631      3452
           1     0.7730    0.7401    0.7562      3502

    accuracy                         0.7597      6954
   macro avg     0.7602    0.7598    0.7597      6954
weighted avg     0.7603    0.7597    0.7596      6954



In [8]:
# Fit on all training data, predict the test set
model.fit(X, y)
test_preds = model.predict(X_test)

In [9]:
submission = pd.DataFrame({
    ID_COL: test[ID_COL],
    TARGET: test_preds.astype(int),
})
submission.to_csv("submission.csv", index=False)

print("Wrote submission.csv", submission.shape)
print(submission[TARGET].value_counts().to_dict())
submission.head()

Wrote submission.csv (1739, 2)
{0: 876, 1: 863}


,GuestID,Churned
0,154038,1
1,620160,0
2,655103,0
3,126993,0
4,635228,0
